In [ ]:
nums = [1,2,4,6,8]
target = 5


for i in range(0,len(nums)):
    for j in range(0,len(nums)):
        if nums[i] + nums[j] == target:
        
            print(nums[i],nums[j])
        break


4 1


In [ ]:
nus = [1,3,5,2,7,8]
tar = 15

dic = {}

for i,j in enumerate (nus):
    min = tar - j
    if min in dic:
        print(dic[min],i)
    else:
        dic[j] = i

4 5


In [ ]:
def load_img():
    img = cv2.imread('/content/bricks.jpg')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

In [ ]:
# Complete Neural Network / Machine Learning Implementation from Scratch
import math
import random

# Seed for reproducibility
random.seed(42)

# --- 1. Matrix and Vector Math Utilities ---

class Matrix:
    def __init__(self, rows, cols, fill=0.0):
        self.rows = rows
        self.cols = cols
        self.data = [[fill for _ in range(cols)] for _ in range(rows)]

    @classmethod
    def random(cls, rows, cols):
        m = cls(rows, cols)
        for i in range(rows):
            for j in range(cols):
                m.data[i][j] = random.gauss(0.0, 1.0 * math.sqrt(2.0 / rows))
        return m

    def __matmul__(self, other):
        if self.cols != other.rows:
            raise ValueError(f"Incompatible dimensions for multiplication: {self.cols} vs {other.rows}")
        result = Matrix(self.rows, other.cols)
        for i in range(self.rows):
            for k in range(self.cols):
                vik = self.data[i][k]
                res_i = result.data[i]
                oth_k = other.data[k]
                for j in range(other.cols):
                    res_i[j] += vik * oth_k[j]
        return result

    def transpose(self):
        result = Matrix(self.cols, self.rows)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[j][i] = self.data[i][j]
        return result

    def apply(self, func):
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = func(self.data[i][j])
        return result

# --- 2. Activation Functions ---

def relu(x):
    return max(0.0, x)

def relu_derivative(x):
    return 1.0 if x > 0.0 else 0.0

def sigmoid(x):
    if x < -709.0: return 0.0
    if x > 709.0: return 1.0
    return 1.0 / (1.0 + math.exp(-x))

# --- 3. Neural Network Layers ---

class DenseLayer:
    def __init__(self, input_size, output_size, activation="relu"):
        self.weights = Matrix.random(input_size, output_size)
        self.bias = Matrix(1, output_size, fill=0.0)
        self.activation = activation
        self.input = None
        self.output = None
        self.grad_weights = Matrix(input_size, output_size)
        self.grad_bias = Matrix(1, output_size)

    def forward(self, input_matrix):
        self.input = input_matrix
        # Z = Input @ Weights + Bias
        self.z = Matrix(input_matrix.rows, self.weights.cols)
        for i in range(input_matrix.rows):
            for j in range(self.weights.cols):
                val = self.bias.data[0][j]
                for k in range(input_matrix.cols):
                    val += input_matrix.data[i][k] * self.weights.data[k][j]
                self.z.data[i][j] = val
        
        # Apply activation
        self.output = Matrix(self.z.rows, self.z.cols)
        for i in range(self.z.rows):
            for j in range(self.z.cols):
                if self.activation == "relu":
                    self.output.data[i][j] = relu(self.z.data[i][j])
                elif self.activation == "sigmoid":
                    self.output.data[i][j] = sigmoid(self.z.data[i][j])
                else:
                    self.output.data[i][j] = self.z.data[i][j] # linear
        return self.output

    def backward(self, output_gradient, lr):
        # Compute derivative of activation
        dz = Matrix(self.z.rows, self.z.cols)
        for i in range(self.z.rows):
            for j in range(self.z.cols):
                if self.activation == "relu":
                    dz.data[i][j] = output_gradient.data[i][j] * relu_derivative(self.z.data[i][j])
                elif self.activation == "sigmoid":
                    sig = sigmoid(self.z.data[i][j])
                    dz.data[i][j] = output_gradient.data[i][j] * sig * (1.0 - sig)
                else:
                    dz.data[i][j] = output_gradient.data[i][j]

        # Gradients for weights and bias: dW = Input^T @ dz
        self.grad_weights = self.input.transpose() @ dz
        self.grad_bias = Matrix(1, dz.cols)
        for j in range(dz.cols):
            col_sum = sum(dz.data[i][j] for i in range(dz.rows))
            self.grad_bias.data[0][j] = col_sum

        # Gradient for previous layer: dx = dz @ Weights^T
        input_gradient = dz @ self.weights.transpose()

        # Update weights and bias (SGD step)
        for i in range(self.weights.rows):
            for j in range(self.weights.cols):
                self.weights.data[i][j] -= lr * self.grad_weights.data[i][j]
        for j in range(self.bias.cols):
            self.bias.data[0][j] -= lr * self.grad_bias.data[0][j]

        return input_gradient

# --- 4. Model and Training Loop ---

class SimpleMLP:
    def __init__(self, layer_sizes):
        self.layers = []
        for i in range(len(layer_sizes) - 1):
            act = "relu" if i < len(layer_sizes) - 2 else "sigmoid"
            self.layers.append(DenseLayer(layer_sizes[i], layer_sizes[i+1], activation=act))

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def train(self, X, y, epochs=100, lr=0.01):
        for epoch in range(epochs):
            total_loss = 0.0
            for r in range(X.rows):
                # Extract single row samples
                x_single = Matrix(1, X.cols)
                x_single.data[0] = list(X.data[r])
                y_single = Matrix(1, y.cols)
                y_single.data[0] = list(y.data[r])

                # Forward pass
                pred = self.forward(x_single)

                # Compute MSE Loss and gradient
                loss_grad = Matrix(1, pred.cols)
                for j in range(pred.cols):
                    diff = pred.data[0][j] - y_single.data[0][j]
                    total_loss += diff ** 2
                    loss_grad.data[0][j] = 2.0 * diff / pred.rows

                # Backward pass
                grad = loss_grad
                for layer in reversed(self.layers):
                    grad = layer.backward(grad, lr)

            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss / X.rows:.5f}")

# --- 5. Generate Synthetic Data and Run ---

if __name__ == "__main__":
    # Create XOR dataset
    X = Matrix(4, 2)
    X.data = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
    
    y = Matrix(4, 1)
    y.data = [[0.0], [1.0], [1.0], [0.0]]

    print("Initializing Neural Network (Architecture: 2 -> 4 -> 1)...")
    model = SimpleMLP([2, 4, 1])

    print("Starting training...")
    model.train(X, y, epochs=200, lr=0.1)

    print("\nTesting predictions after training:")
    for r in range(X.rows):
        x_single = Matrix(1, 2)
        x_single.data[0] = list(X.data[r])
        p = model.forward(x_single)
        print(f"Input: {X.data[r]} | Target: {y.data[r]} | Prediction: {p.data[0][j]:.4f}" if 'j' in locals() else f"Input: {X.data[r]} | Target: {y.data[r]} | Prediction: {p.data[0][0]:.4f}")


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Model

# Data generation ke liye seed set karein
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)

# =====================================================================
# 1. ARTIFICIAL NEURAL NETWORK (ANN) - For Tabular Data
# =====================================================================
print("\n--- 1. Building and Training ANN ---")
# Dummy Tabular Data: 1000 samples, 20 features, 3 classes
X_ann = np.random.randn(1000, 20)
y_ann = np.random.randint(0, 3, size=(1000,))

ann_model = models.Sequential([
    layers.Input(shape=(20,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax')  # Multi-class classification
])

ann_model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

# Train ANN
ann_model.fit(X_ann, y_ann, epochs=5, batch_size=32, verbose=1)


# =====================================================================
# 2. CONVOLUTIONAL NEURAL NETWORK (CNN) - For Image Data
# =====================================================================
print("\n--- 2. Building and Training CNN ---")
# Dummy Image Data: 500 images, 32x32 pixels, 3 channels (RGB), 10 classes
X_cnn = np.random.randn(500, 32, 32, 3)
y_cnn = np.random.randint(0, 10, size=(500,))

cnn_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

# Train CNN
cnn_model.fit(X_cnn, y_cnn, epochs=5, batch_size=32, verbose=1)


# =====================================================================
# 3. RECURRENT NEURAL NETWORK (RNN) - For Time-Series/Sequence Data
# =====================================================================
print("\n--- 3. Building and Training Simple RNN ---")
# Dummy Sequence Data: 400 samples, timesteps=10, features=8, binary target
X_rnn = np.random.randn(400, 10, 8)
y_rnn = np.random.randint(0, 2, size=(400,))

rnn_model = models.Sequential([
    layers.Input(shape=(10, 8)),
    layers.SimpleRNN(32, activation='tanh', return_sequences=False),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

rnn_model.compile(optimizer='adam', 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])

# Train RNN
rnn_model.fit(X_rnn, y_rnn, epochs=5, batch_size=32, verbose=1)


# =====================================================================
# 4. LONG SHORT-TERM MEMORY (LSTM) - For Long Sequences/Text Data
# =====================================================================
print("\n--- 4. Building and Training LSTM ---")
# Dummy Sequence Data: 400 samples, timesteps=15, features=10
X_lstm = np.random.randn(400, 15, 10)
y_lstm = np.random.randint(0, 2, size=(400,))

lstm_model = models.Sequential([
    layers.Input(shape=(15, 10)),
    layers.LSTM(64, return_sequences=True),  # Returns sequence for next LSTM
    layers.LSTM(32),                         # Final hidden state
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy'])

# Train LSTM
lstm_model.fit(X_lstm, y_lstm, epochs=5, batch_size=32, verbose=1)


# =====================================================================
# 5. TRANSFORMER MODEL (Self-Attention) - For Advanced NLP/Sequences
# =====================================================================
print("\n--- 5. Building and Training Transformer Layer ---")
# Dummy Sequence Data: 400 samples, timesteps=20, embedding_dim=16
X_trans = np.random.randn(400, 20, 16)
y_trans = np.random.randint(0, 5, size=(400,))

# Custom Transformer Block using Keras Functional API
inputs = layers.Input(shape=(20, 16))

# Self-Attention Layer
attn_output = layers.MultiHeadAttention(num_heads=2, key_dim=16)(inputs, inputs)
attn_output = layers.Dropout(0.1)(attn_output)
out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)

# Feed Forward Layer
ffn_output = layers.Dense(32, activation="relu")(out1)
ffn_output = layers.Dense(16)(ffn_output)
ffn_output = layers.Dropout(0.1)(ffn_output)
sequence_output = layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

# Classification Heads
pooling = layers.GlobalAveragePooling1D()(sequence_output)
outputs = layers.Dense(5, activation="softmax")(pooling)

transformer_model = Model(inputs=inputs, outputs=outputs)

transformer_model.compile(optimizer='adam', 
                          loss='sparse_categorical_crossentropy', 
                          metrics=['accuracy'])

# Train Transformer
transformer_model.fit(X_trans, y_trans, epochs=5, batch_size=32, verbose=1)

print("\n[SUCCESS] Sabhi models ka pipeline pipeline successfully run ho gaya hai!")


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# =====================================================================
# 1. DUMMY DATA CREATION (Imagine this is your custom dataset)
# =====================================================================
# Pre-trained models expect specific sizes (e.g., MobileNetV2 expects 224x224x3)
print("Generating dummy image dataset (224x224)...")
X_train = np.random.uniform(0, 255, size=(200, 224, 224, 3)).astype(np.float32)
y_train = np.random.randint(0, 5, size=(200,))  # 5 custom classes (e.g., Apple, Banana, Orange, etc.)

# =====================================================================
# 2. LOAD PRE-TRAINED BASE MODEL (Transfer Learning)
# =====================================================================
print("\nLoading Pre-trained MobileNetV2 base model...")
# Include_top=False matlab ImageNet ke original 1000-class classifier head ko hata dena
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Phase 1: Base model ke weights ko FREEZE karein taaki humara new head seekh sake
base_model.trainable = False

# =====================================================================
# 3. ATTACH CUSTOM CLASSIFICATION HEAD
# =====================================================================
inputs = layers.Input(shape=(224, 224, 3))

# MobileNetV2 specific preprocessing wrapper
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)  # training=False rules out BatchNormalization updates

# Pooling & Dense Layers for our 5 custom classes
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)  # Overfitting se bachne ke liye dropout
outputs = layers.Dense(5, activation='softmax')(x)  # 5 classes output

model = models.Model(inputs, outputs)

# =====================================================================
# PHASE 1: WARM-UP TRAINING (Only Train the Top Head)
# =====================================================================
print("\n--- Phase 1: Training Custom Classification Head Only ---")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Base model frozen hai, sirf upar ki lagayi hui layers train hongi
model.fit(X_train, y_train, epochs=3, batch_size=16)

# =====================================================================
# PHASE 2: ACTUAL FINE-TUNING (Unfreeze Base Layers + Very Low LR)
# =====================================================================
print("\n--- Phase 2: Fine-Tuning Top Layers of Base Model ---")
# Base model ko unfreeze karein
base_model.trainable = True

# Par dhyan rahe, poore layers ko tune nahi karte, end ki kuch layers ko karte hain.
# Let's see how many layers are there and freeze everything except the last 20 layers.
print(f"Total layers in base model: {len(base_model.layers)}")
fine_tune_at = len(base_model.layers) - 20

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# IMPORTANT: Fine-tuning ke waqt Learning Rate ko bohot kam (10x-100x lower) rakhte hain
# taaki pre-trained weights destroy na ho jayein (Catastrophic Forgetting se bachne ke liye)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), # Low learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Start actual fine-tuning
model.fit(X_train, y_train, epochs=3, batch_size=16)

print("\n[SUCCESS] Model successfully fine-tuned for your custom classes!")


In [ ]:
# RAG (Retrieval-Augmented Generation) End-to-End Implementation
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Initializing RAG components...")

# =====================================================================
# 1. KNOWLEDGE BASE (Documents)
# =====================================================================
# Yeh hamara private/custom data hai jisse LLM ko jawab dena hai
documents = [
    "Indore is the cleanest city in India and is located in the state of Madhya Pradesh.",
    "The capital of Madhya Pradesh is Bhopal, but Indore is considered the commercial capital.",
    "Poha-Jalebi is the most famous breakfast dish in Indore.",
    "Rajwada and 52 Dukan are popular tourist and food spots in Indore.",
    "TensorFlow and PyTorch are popular deep learning frameworks used for building AI models."
]

# =====================================================================
# 2. EMBEDDING MODEL & VECTOR DATABASE (Retrieval Part)
# =====================================================================
# Sentence-Transformers ka use karke text ko vectors (numbers) me convert karenge
print("Loading embedding model (Sentence-Transformers)...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# ChromaDB client setup (In-memory database)
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="my_knowledge_base")

# Documents ko embed karke ChromaDB me store karna
print("Storing documents into ChromaDB...")
embeddings = embed_model.encode(documents).tolist()
ids = [f"doc_{i}" for i in range(len(documents))]

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids
)

# =====================================================================
# 3. GENERATION MODEL (LLM Part)
# =====================================================================
# Hugging Face se ek chota, fast local LLM load kar rahe hain generation ke liye
print("Loading local LLM pipeline (Flan-T5)...")
generator = pipeline("text2text-generation", model="google/flan-t5-small", max_length=128)

# =====================================================================
# 4. RAG PIPELINE FUNCTION (Retrieve + Generate)
# =====================================================================
def ask_rag(query, n_results=2):
    print(f"\nUser Query: '{query}'")
    
    # Step A: RETRIEVAL - Query ki embedding banao aur database me search karo
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    
    # Retrieved chunks ko nikalna
    retrieved_docs = results['documents']
    context = "\n".join(retrieved_docs)
    print(f"\n[Retrieved Context from DB]:\n{context}")
    
    # Step B: AUGMENTATION & GENERATION - Context aur Query ko milakar LLM ko do
    prompt = f"Based on the following context, answer the question accurately.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    print("Generating response from LLM...")
    response = generator(prompt)
    
    return response['generated_text']

# =====================================================================
# 5. TEST THE RAG SYSTEM
# =====================================================================
if __name__ == "__main__":
    # Test query jo hamare documents par based hai
    user_query = "What is the most famous breakfast in Indore?"
    answer = ask_rag(user_query)
    
    print(f"\n[Final RAG Answer]: {answer}")
    
    # Ek aur test query
    user_query_2 = "Tell me about deep learning frameworks."
    answer_2 = ask_rag(user_query_2)
    
    print(f"\n[Final RAG Answer]: {answer_2}")


In [ ]:
# Autonomous AI Agent Implementation (Reason + Act Pattern)
import re
import math

# =====================================================================
# 1. DEFINE TOOLS FOR THE AGENT
# =====================================================================
def get_calculator(expression):
    """Calculates mathematical expressions safely."""
    try:
        # Safe evaluation of basic math
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in math expression."
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error in calculation: {str(e)}"

def get_weather(city):
    """Simulates fetching weather data for a city."""
    weather_data = {
        "indore": "28°C, Partly Cloudy with a pleasant breeze.",
        "bhopal": "29°C, Sunny.",
        "mumbai": "32°C, Humid and Rainy."
    }
    city_clean = city.strip().lower()
    return weather_data.get(city_clean, f"Sorry, weather data for '{city}' is not available.")

# Map tool names to actual functions
tools = {
    "calculator": get_calculator,
    "weather": get_weather
}

# =====================================================================
# 2. AGENT REASONING & EXECUTION ENGINE (ReAct Loop)
# =====================================================================
class SimpleAIAgent:
    def __init__(self, tools_dict):
        self.tools = tools_dict

    def think_and_act(self, query):
        print(f"\n[Agent received query]: {query}")
        
        # Simple rule-based reasoning simulation (Acting as LLM's brain)
        # In production, you would call OpenAI/Gemini API here to generate Thought/Action
        action, action_input = self._mock_llm_reasoning(query)
        
        if action == "final_answer":
            return action_input
        
        # Execute the selected tool
        if action in self.tools:
            print(f"[Agent Action]: Calling tool '{action}' with input '{action_input}'")
            tool_result = self.tools[action](action_input)
            print(f"[Tool Output]: {tool_result}")
            
            # Final synthesis based on tool output
            final_response = f"Based on the tool output for '{action_input}', the answer is: {tool_result}"
            return final_response
        else:
            return "I am sorry, I couldn't find an appropriate tool to handle this request."

    def _mock_llm_reasoning(self, query):
        """Simulates LLM deciding which tool to call based on keywords."""
        q_lower = query.lower()
        
        if "weather" in q_lower or "temperature" in q_lower:
            # Extract city name roughly
            for city in ["indore", "bhopal", "mumbai"]:
                if city in q_lower:
                    return "weather", city
            return "weather", "indore" # Default
            
        elif any(op in query for op in ["+", "-", "*", "/", "calculate", "math"]):
            # Extract math expression
            match = re.search(r'[\d\+\-\*\/\.\(\)\s]+', query)
            if match:
                return "calculator", match.group().strip()
                
        # Default fallback to direct conversational answer
        return "final_answer", f"I processed your request directly: '{query}'"

# =====================================================================
# 3. RUNNING THE AI AGENT
# =====================================================================
if __name__ == "__main__":
    agent = SimpleAIAgent(tools_dict=tools)
    
    # Test 1: Weather Query
    response_1 = agent.think_and_act("What is the weather like in Indore today?")
    print(f"\n[Agent Final Response]: {response_1}")
    
    # Test 2: Math Calculation Query
    response_2 = agent.think_and_act("Can you calculate 456 * 123 for me?")
    print(f"\n[Agent Final Response]: {response_2}")
